# Phase 5 v2 — Pairwise Cross-Encoder Reranking

This notebook extends the executed Phase 4 notebook with a second-stage reranker:

```text
new issue
   ↓
Phase 3 fine-tuned MiniLM bi-encoder
   ↓
strictly older Top-50 candidates
   ↓
fine-tuned cross-encoder jointly compares each positive against hard negatives
   ↓
reranked Top-10 duplicate suggestions
```

The primary experiment deliberately uses **dense Top-50 only**. This isolates the
effect of cross-encoder reranking from Phase 4 RRF. A reranker cannot recover a
relevant issue absent from Top-50, so candidate Hit@50 is reported as its oracle
coverage ceiling.

Leakage protocol:

1. Mine hard negatives and train a grouped pairwise RankNet objective using `dev_fit` only.
2. Select the reranker using the chronological validation tail only.
3. If it wins, restart from the pinned base model and retrain on full official train.
4. Official test is disabled by default.

Important: Phase 4 in the source notebook already opened the official test. Therefore
any Phase 5 score on that same test is an **exploratory reused-holdout result**, not a
pristine confirmatory result. A new later temporal holdout is required for the
strongest final claim.

Version 2 replaces independent BCE classification with a query-grouped pairwise
objective: `softplus(-(score_positive - score_negative))`. BCE is a legitimate
cross-encoder loss, but the observed validation degradation motivates an objective
that directly matches reranking. Random/easy negatives are disabled by default.


In [ ]:
import gc
import hashlib
import html
import json
import math
import os
import platform
import random
import re
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import torch
    import sentence_transformers
    import transformers
    from sentence_transformers import SentenceTransformer
    from transformers import (
        AutoModelForSequenceClassification,
        AutoTokenizer,
        get_linear_schedule_with_warmup,
    )
except ImportError:
    print("Installing sentence-transformers and transformers...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "sentence-transformers", "transformers",
    ])
    import torch
    import sentence_transformers
    import transformers
    from sentence_transformers import SentenceTransformer
    from transformers import (
        AutoModelForSequenceClassification,
        AutoTokenizer,
        get_linear_schedule_with_warmup,
    )

from torch.utils.data import DataLoader, Dataset

try:
    from IPython.display import display
except ImportError:
    display = print

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 180)

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("transformers:", transformers.__version__)
print("sentence-transformers:", sentence_transformers.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


## 1. Paths, frozen configuration, and safety switches

`cross-encoder/ms-marco-MiniLM-L6-v2` is initialized at a pinned Hugging Face
revision. It is a ranking cross-encoder, not the Phase 3 bi-encoder checkpoint;
the two architectures have different roles.

All expensive switches are `False`. Run development sections first. Do not enable
final retraining until validation writes a winning selection artifact. The reused
official test requires a second explicit acknowledgement switch.

In [ ]:
# ===== KAGGLE PATHS =====
DATA_ROOT = Path(
    "/kaggle/input/datasets/nguyendinhtann/duplicatedetection/DuplicateDetection_Kaggle"
)
PHASE3_INPUT_ROOT = DATA_ROOT / "phase3_minilm_finetune_outputs"
PHASE4_INPUT_ROOT = DATA_ROOT / "phase4_rrf_outputs"
PHASE5_INPUT_ROOT = DATA_ROOT / "phase5_cross_encoder_outputs"

OUTPUT_ROOT = Path("/kaggle/working/phase5_cross_encoder_outputs")
CACHE_ROOT = OUTPUT_ROOT / "cache"
DEV_CE_OUTPUT = OUTPUT_ROOT / "dev_cross_encoder" / "final"
FINAL_CE_OUTPUT = OUTPUT_ROOT / "final_cross_encoder" / "final"
for path in (OUTPUT_ROOT, CACHE_ROOT):
    path.mkdir(parents=True, exist_ok=True)

DEV_DENSE_CHECKPOINT = PHASE3_INPUT_ROOT / "dev_model" / "final"
FINAL_DENSE_CHECKPOINT = PHASE3_INPUT_ROOT / "final_model" / "final"

PROJECTS = {
    "Firefox": {
        "main": DATA_ROOT / "Firefox" / "mozilla_firefox.csv",
        "train": DATA_ROOT / "Firefox" / "train.csv",
        "test": DATA_ROOT / "Firefox" / "test.csv",
    },
    "MozillaCore": {
        "main": DATA_ROOT / "MozillaCore" / "mozilla_core.csv",
        "train": DATA_ROOT / "MozillaCore" / "train.csv",
        "test": DATA_ROOT / "MozillaCore" / "test.csv",
    },
}

VALIDATION_FRACTION_OF_OFFICIAL_TRAIN = 0.15
METRIC_KS = (1, 5, 10)
CANDIDATE_DEPTH = 50
OUTPUT_K = 10

DENSE_DESCRIPTION_CHAR_LIMIT = 4_000
DENSE_TEXT_VERSION = f"title-newline-description-head-{DENSE_DESCRIPTION_CHAR_LIMIT}"
DENSE_ENCODE_BATCH_SIZE = 128
DENSE_QUERY_BATCH_SIZE = 64

CE_BASE_MODEL = "cross-encoder/ms-marco-MiniLM-L6-v2"
CE_BASE_REVISION = "c5ee24cb16019beea0893ab7796b1df96625c6b8"
# If Kaggle internet is disabled, add the pinned model as an input dataset and set this.
CE_BASE_LOCAL_PATH = None
CE_DESCRIPTION_CHAR_LIMIT = 2_500
CE_MAX_LENGTH = 512
CE_QUERY_BATCH_SIZE = 4
CE_INFERENCE_BATCH_SIZE = 128
CE_EPOCHS = 1
CE_LEARNING_RATE = 1e-5
CE_WEIGHT_DECAY = 0.01
CE_WARMUP_RATIO = 0.10
CE_GRADIENT_ACCUMULATION = 4
PAIRWISE_SIGMA = 1.0

MAX_POSITIVES_PER_QUERY = 1
HARD_NEGATIVE_SEARCH_DEPTH = 100
HARD_NEGATIVES_PER_QUERY = 3
RANDOM_NEGATIVES_PER_QUERY = 0

DENSE_PROGRESS_EVERY = 1_024
PAIR_ASSEMBLY_PROGRESS_EVERY = 5_000
FORCE_REBUILD_PAIR_DATA = False

SELECTION_AGGREGATION = "macro_project_average"
PRIMARY_SELECTION_METRIC = "hit@10"
SELECTION_TIE_BREAKERS = ("mrr@10", "recall@10")

# Development pipeline: enable in order.
RUN_BUILD_DEV_TRAINING_DATA = False
RUN_TRAIN_DEV_CROSS_ENCODER = False
RUN_PRETRAINED_CE_VALIDATION = False
RUN_DEV_VALIDATION_RERANK = False

# Enable only after a winning complete-validation selection exists.
RUN_BUILD_FINAL_TRAINING_DATA = False
RUN_FINAL_RETRAIN = False

# Phase 4 already evaluated this test. Explicit acknowledgement is mandatory.
OFFICIAL_TEST_WAS_PREVIOUSLY_OPENED_IN_PHASE4 = True
ALLOW_REUSED_OFFICIAL_TEST_FOR_EXPLORATORY_REPORT = False
RUN_OFFICIAL_TEST = False

FORCE_REBUILD_DENSE_EMBEDDINGS = False
FORCE_REBUILD_RANKINGS = False

for project, paths in PROJECTS.items():
    for kind, path in paths.items():
        assert path.exists(), f"Missing {project}/{kind}: {path}"
assert DEV_DENSE_CHECKPOINT.exists(), DEV_DENSE_CHECKPOINT
assert FINAL_DENSE_CHECKPOINT.exists(), FINAL_DENSE_CHECKPOINT
assert CANDIDATE_DEPTH >= OUTPUT_K >= max(METRIC_KS)
assert HARD_NEGATIVE_SEARCH_DEPTH >= CANDIDATE_DEPTH
assert CE_EPOCHS > 0 and CE_QUERY_BATCH_SIZE > 0

print("DATA_ROOT:", DATA_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("Dense checkpoints:", DEV_DENSE_CHECKPOINT.exists(), FINAL_DENSE_CHECKPOINT.exists())
print("CE base:", CE_BASE_LOCAL_PATH or f"{CE_BASE_MODEL}@{CE_BASE_REVISION}")
print("All run switches default to False:", not any([
    RUN_BUILD_DEV_TRAINING_DATA, RUN_TRAIN_DEV_CROSS_ENCODER,
    RUN_PRETRAINED_CE_VALIDATION, RUN_DEV_VALIDATION_RERANK,
    RUN_BUILD_FINAL_TRAINING_DATA,
    RUN_FINAL_RETRAIN, RUN_OFFICIAL_TEST,
]))


def phase5_artifact_for_read(relative_path):
    """Prefer this run's artifact, then an uploaded Phase 5 Kaggle input."""
    relative_path = Path(relative_path)
    candidates = [OUTPUT_ROOT / relative_path, PHASE5_INPUT_ROOT / relative_path]
    return next((path for path in candidates if path.exists()), candidates[0])


## 2. Load exactly the Phase 4 data schema

The same issue IDs, text cleaning, chronological ordering, official train/test
files, and multi-relevant `Duplicate` parsing are retained. `ce_text` is separate
from dense `embedding_text`; the cross-encoder tokenizer applies longest-first
pair truncation at 512 tokens.

In [ ]:
def normalize_id(value):
    if pd.isna(value):
        return None
    return re.sub(r"\.0$", "", str(value).strip())


def normalize_text(value):
    if pd.isna(value):
        return ""
    text = html.unescape(str(value))
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"https?://\S+|www\.\S+", " <URL> ", text)
    return re.sub(r"\s+", " ", text).strip()


def parse_duplicate_set(value):
    if pd.isna(value):
        return frozenset()
    text = str(value).strip()
    if not text or text.upper() == "NULL":
        return frozenset()
    return frozenset(
        normalized for item in text.split(";")
        if (normalized := normalize_id(item))
    )


def load_project(project, paths):
    issues = pd.read_csv(paths["main"], low_memory=False).rename(columns={
        "Issue_id": "issue_id", "Priority": "priority",
        "Component": "component", "Duplicated_issue": "duplicate_of",
        "Title": "title", "Description": "description",
        "Resolution": "resolution", "Created_time": "created_at",
    })
    required = {
        "issue_id", "title", "description", "created_at",
        "duplicate_of", "component", "resolution",
    }
    assert required.issubset(issues.columns), required - set(issues.columns)
    issues["issue_id"] = issues["issue_id"].map(normalize_id)
    issues["created_at"] = pd.to_datetime(issues["created_at"], errors="coerce", utc=True)
    issues["title_clean"] = issues["title"].map(normalize_text)
    issues["description_clean"] = issues["description"].map(normalize_text)
    issues["text"] = (
        issues["title_clean"] + " . " + issues["title_clean"]
        + " . " + issues["description_clean"]
    ).str.strip()
    issues["embedding_text"] = (
        issues["title_clean"] + "\n"
        + issues["description_clean"].str.slice(0, DENSE_DESCRIPTION_CHAR_LIMIT)
    ).str.strip()
    issues["ce_text"] = (
        "[TITLE] " + issues["title_clean"] + "\n[DESCRIPTION] "
        + issues["description_clean"].str.slice(0, CE_DESCRIPTION_CHAR_LIMIT)
    ).str.strip()
    issues["project"] = project
    assert issues["issue_id"].notna().all()
    assert not issues["issue_id"].duplicated().any()
    assert issues["created_at"].notna().all()
    issues = issues.sort_values(["created_at", "issue_id"]).reset_index(drop=True)
    issues["position"] = np.arange(len(issues), dtype=np.int64)

    splits = {}
    for split in ("train", "test"):
        labels = pd.read_csv(paths[split], dtype=str)
        assert {"Issue_id", "Duplicate"}.issubset(labels.columns)
        labels = labels.rename(columns={"Issue_id": "issue_id", "Duplicate": "duplicate_raw"})
        labels["issue_id"] = labels["issue_id"].map(normalize_id)
        labels["relevant_ids"] = labels["duplicate_raw"].map(parse_duplicate_set)
        assert labels["issue_id"].notna().all()
        assert not labels["issue_id"].duplicated().any()
        splits[split] = labels[["issue_id", "relevant_ids"]].copy()

    all_ids = set(issues["issue_id"])
    train_ids = set(splits["train"]["issue_id"])
    test_ids = set(splits["test"]["issue_id"])
    assert train_ids.isdisjoint(test_ids)
    assert train_ids | test_ids == all_ids
    id_to_time = issues.set_index("issue_id")["created_at"].to_dict()
    id_to_text = issues.set_index("issue_id")["text"].to_dict()
    id_to_ce_text = issues.set_index("issue_id")["ce_text"].to_dict()
    return {
        "project": project, "issues": issues,
        "train_labels": splits["train"], "test_labels": splits["test"],
        "official_train_ids": train_ids, "official_test_ids": test_ids,
        "id_to_time": id_to_time, "id_to_text": id_to_text,
        "id_to_ce_text": id_to_ce_text,
    }


datasets = {project: load_project(project, paths) for project, paths in PROJECTS.items()}
display(pd.DataFrame([
    {
        "project": project, "issues": len(data["issues"]),
        "official_train": len(data["train_labels"]),
        "official_test": len(data["test_labels"]),
        "empty_text": int(data["issues"]["text"].str.len().eq(0).sum()),
    }
    for project, data in datasets.items()
]))


## 3. Chronological validation and test queries

Only strictly older relevant reports are retained. Validation is the last 15% of
official train by time. Earlier validation issues may appear as candidates for a
later validation query (a realistic rolling index), but no validation labels are
used for training or negative mining.

In [ ]:
def filter_temporal_queries(data, labels, allowed_query_ids=None):
    allowed_query_ids = None if allowed_query_ids is None else set(allowed_query_ids)
    rows = []
    audit = {
        "labeled_queries": 0, "empty_query_text": 0,
        "no_historical_relevant": 0, "removed_relevance_links": 0,
    }
    for query_id, raw_relevant in labels[["issue_id", "relevant_ids"]].itertuples(index=False):
        if allowed_query_ids is not None and query_id not in allowed_query_ids:
            continue
        if not raw_relevant:
            continue
        audit["labeled_queries"] += 1
        if not data["id_to_text"].get(query_id, ""):
            audit["empty_query_text"] += 1
            continue
        query_time = data["id_to_time"][query_id]
        relevant = {
            target for target in raw_relevant
            if target in data["id_to_time"]
            and data["id_to_text"].get(target, "")
            and data["id_to_time"][target] < query_time
        }
        audit["removed_relevance_links"] += len(raw_relevant) - len(relevant)
        if not relevant:
            audit["no_historical_relevant"] += 1
            continue
        rows.append({
            "query_id": query_id, "query_time": query_time,
            "relevant_ids": frozenset(relevant), "num_relevant": len(relevant),
        })
    frame = pd.DataFrame(rows)
    if len(frame):
        frame = frame.sort_values(["query_time", "query_id"]).reset_index(drop=True)
    audit["evaluable_queries"] = len(frame)
    return frame, audit


query_sets, query_audits = {}, []
for project, data in datasets.items():
    issues = data["issues"]
    train_issues = issues[issues["issue_id"].isin(data["official_train_ids"])]
    train_issues = train_issues.sort_values(["created_at", "issue_id"])
    validation_size = math.ceil(len(train_issues) * VALIDATION_FRACTION_OF_OFFICIAL_TRAIN)
    validation_ids = set(train_issues.tail(validation_size)["issue_id"])
    dev_fit_ids = set(train_issues.iloc[:-validation_size]["issue_id"])
    validation, validation_audit = filter_temporal_queries(
        data, data["train_labels"], validation_ids
    )
    test, test_audit = filter_temporal_queries(data, data["test_labels"])

    dev_max = issues.loc[issues["issue_id"].isin(dev_fit_ids), "created_at"].max()
    validation_min = issues.loc[issues["issue_id"].isin(validation_ids), "created_at"].min()
    test_min = issues.loc[issues["issue_id"].isin(data["official_test_ids"]), "created_at"].min()
    official_train_max = issues.loc[
        issues["issue_id"].isin(data["official_train_ids"]), "created_at"
    ].max()
    assert dev_max < validation_min
    assert official_train_max < test_min
    assert all(
        data["id_to_time"][target] < row.query_time
        for row in validation.itertuples() for target in row.relevant_ids
    )
    assert all(
        data["id_to_time"][target] < row.query_time
        for row in test.itertuples() for target in row.relevant_ids
    )
    query_sets[project] = {
        "dev_fit_ids": dev_fit_ids, "validation_ids": validation_ids,
        "validation": validation, "test": test,
    }
    query_audits.extend([
        {"project": project, "split": "validation", **validation_audit},
        {"project": project, "split": "test", **test_audit},
    ])

query_audits = pd.DataFrame(query_audits)
display(query_audits)
print("TEMPORAL DATA GATE: PASS")


## 4. Dense candidate generator and metrics

The exact Phase 3 development checkpoint generates validation candidates. The
final Phase 3 checkpoint generates official-test candidates. Every candidate is
strictly older than its query, and deterministic rank ties use corpus position.

In [ ]:
def ranking_metrics(retrieved_ids, relevant_ids, ks=METRIC_KS):
    relevant = set(relevant_ids)
    result = {}
    for k in ks:
        hits = len(set(retrieved_ids[:k]) & relevant)
        result[f"hit@{k}"] = float(hits > 0)
        result[f"recall@{k}"] = hits / len(relevant)
    first_rank = next(
        (rank for rank, issue_id in enumerate(retrieved_ids[:10], start=1)
         if issue_id in relevant), None,
    )
    result["mrr@10"] = 0.0 if first_rank is None else 1.0 / first_rank
    dcg = sum(
        1.0 / math.log2(rank + 1)
        for rank, issue_id in enumerate(retrieved_ids[:10], start=1)
        if issue_id in relevant
    )
    ideal_hits = min(len(relevant), 10)
    idcg = sum(1.0 / math.log2(rank + 1) for rank in range(1, ideal_hits + 1))
    result["ndcg@10"] = dcg / idcg if idcg else 0.0
    return result


def top_k_indices(scores, k):
    k = min(k, len(scores))
    if k == 0:
        return np.array([], dtype=np.int64)
    candidates = np.arange(len(scores)) if k == len(scores) else np.argpartition(-scores, k - 1)[:k]
    order = np.lexsort((candidates, -scores[candidates]))
    return candidates[order]


def checkpoint_fingerprint(model_path):
    model_path = Path(model_path)
    digest, included = hashlib.sha256(), []
    for path in sorted(model_path.rglob("*")):
        if not path.is_file() or path.suffix not in {".safetensors", ".bin", ".json", ".txt"}:
            continue
        relative = path.relative_to(model_path).as_posix()
        included.append(relative)
        digest.update(relative.encode("utf-8"))
        with path.open("rb") as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b""):
                digest.update(chunk)
    assert included, f"No checkpoint files found under {model_path}"
    return digest.hexdigest()


def encode_dense_project(model, data, model_tag, fingerprint):
    issues = data["issues"]
    digest = hashlib.sha256()
    digest.update(fingerprint.encode("utf-8"))
    digest.update(DENSE_TEXT_VERSION.encode("utf-8"))
    for issue_id, text in issues[["issue_id", "embedding_text"]].itertuples(index=False):
        digest.update(issue_id.encode("utf-8") + b"\0" + text.encode("utf-8"))
    path = CACHE_ROOT / f"{data['project'].lower()}__{model_tag}__{digest.hexdigest()[:20]}.npy"
    if path.exists() and not FORCE_REBUILD_DENSE_EMBEDDINGS:
        matrix = np.load(path)
        assert matrix.shape[0] == len(issues)
        print("DENSE CACHE HIT:", path.name)
        return matrix
    matrix = model.encode(
        issues["embedding_text"].tolist(), batch_size=DENSE_ENCODE_BATCH_SIZE,
        show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True,
    ).astype(np.float32, copy=False)
    np.save(path, matrix)
    return matrix


def retrieve_dense_rolling(data, queries, embeddings, allowed_candidate_ids, depth):
    issues = data["issues"]
    issue_ids = issues["issue_id"].to_numpy()
    created_ns = issues["created_at"].astype("int64").to_numpy()
    id_to_pos = dict(zip(issue_ids, issues["position"]))
    allowed = set(allowed_candidate_ids)
    allowed_mask = np.fromiter((x in allowed for x in issue_ids), dtype=bool, count=len(issue_ids))
    device = "cuda" if torch.cuda.is_available() else "cpu"
    corpus = torch.from_numpy(np.asarray(embeddings)).to(device)
    rows = []
    with torch.inference_mode():
        for start in range(0, len(queries), DENSE_QUERY_BATCH_SIZE):
            stop = min(start + DENSE_QUERY_BATCH_SIZE, len(queries))
            batch = queries.iloc[start:stop]
            positions = np.asarray([id_to_pos[x] for x in batch["query_id"]], dtype=np.int64)
            ends = np.searchsorted(created_ns, batch["query_time"].astype("int64").to_numpy(), side="left")
            assert np.all(ends <= positions)
            query_tensor = corpus[torch.as_tensor(positions, dtype=torch.long, device=device)]
            scores = (query_tensor @ corpus[:int(ends.max())].T).float().cpu().numpy()
            for offset, row in enumerate(batch.itertuples(index=False)):
                end = int(ends[offset])
                legal = np.flatnonzero(allowed_mask[:end])
                selected = legal[top_k_indices(scores[offset, legal], depth)]
                retrieved = issue_ids[selected].tolist()
                assert row.query_id not in retrieved
                assert all(data["id_to_time"][x] < row.query_time for x in retrieved)
                rows.append({
                    "project": data["project"], "query_id": row.query_id,
                    "query_time": row.query_time,
                    "num_candidates": int(len(legal)),
                    "retrieved_ids": ";".join(retrieved),
                })
            if stop % DENSE_PROGRESS_EVERY == 0 or stop == len(queries):
                print(f"  {data['project']} dense: {stop}/{len(queries)}")
    del corpus
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return pd.DataFrame(rows)


def split_ids(value):
    if pd.isna(value) or not str(value).strip():
        return []
    return [x for x in str(value).split(";") if x]


def validate_ranking_frame(frame, data, queries, allowed_ids, expected_depth):
    frame = frame.copy()
    frame["query_id"] = frame["query_id"].map(normalize_id)
    assert not frame["query_id"].duplicated().any()
    assert set(frame["query_id"]) == set(queries["query_id"])
    query_time = queries.set_index("query_id")["query_time"].to_dict()
    allowed = set(allowed_ids)
    for row in frame.itertuples(index=False):
        ids = split_ids(row.retrieved_ids)
        assert len(ids) == len(set(ids))
        assert len(ids) <= expected_depth
        assert set(ids).issubset(allowed)
        assert row.query_id not in ids
        assert all(data["id_to_time"][x] < query_time[row.query_id] for x in ids)
    return frame


assert ranking_metrics(["x", "b"], {"a", "b"})["mrr@10"] == 0.5
assert top_k_indices(np.array([0.1, 0.9, 0.9]), 2).tolist() == [1, 2]
print("RANKING ENGINE TESTS: PASS")


## 5. Build leakage-safe cross-encoder training pairs

Positive pairs are known `(newer query, older duplicate)` links within the chosen
training scope. Hard negatives are high-ranked dense candidates outside the
query's entire known duplicate component. Random negatives, if explicitly enabled for an ablation, are also strictly
older and outside that component. Version 2 defaults to three hard negatives
and zero random negatives because production reranks dense Top-50 candidates.

This exclusion reduces false negatives but cannot eliminate them because duplicate
labels may be incomplete. The notebook records hard/random counts and never mines
from validation or test while building the development dataset.


In [ ]:
class UnionFind:
    def __init__(self, values):
        self.parent = {x: x for x in values}

    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            self.parent[rb] = ra


def known_duplicate_components(data, allowed_ids):
    allowed = set(allowed_ids)
    uf = UnionFind(allowed)
    for query_id, targets in data["train_labels"][["issue_id", "relevant_ids"]].itertuples(index=False):
        if query_id not in allowed:
            continue
        for target in targets:
            if target in allowed:
                uf.union(query_id, target)
    # Main CSV mapping is used only to exclude known duplicate negatives.
    for query_id, raw_target in data["issues"][["issue_id", "duplicate_of"]].itertuples(index=False):
        if query_id not in allowed:
            continue
        for target in parse_duplicate_set(raw_target):
            if target in allowed:
                uf.union(query_id, target)
    groups = {}
    for issue_id in allowed:
        groups.setdefault(uf.find(issue_id), set()).add(issue_id)
    return {issue_id: groups[uf.find(issue_id)] for issue_id in allowed}


def build_positive_pairs(data, allowed_ids):
    allowed = set(allowed_ids)
    rows = []
    for query_id, targets in data["train_labels"][["issue_id", "relevant_ids"]].itertuples(index=False):
        if query_id not in allowed or not data["id_to_ce_text"].get(query_id, ""):
            continue
        query_time = data["id_to_time"][query_id]
        historical = [
            target for target in targets
            if target in allowed and data["id_to_ce_text"].get(target, "")
            and data["id_to_time"][target] < query_time
        ]
        historical.sort(key=lambda x: (data["id_to_time"][x], x), reverse=True)
        for target in historical[:MAX_POSITIVES_PER_QUERY]:
            rows.append({
                "project": data["project"], "query_id": query_id,
                "candidate_id": target, "query_time": query_time,
            })
    frame = pd.DataFrame(rows).drop_duplicates(["query_id", "candidate_id"])
    if len(frame):
        frame = frame.sort_values(["query_time", "query_id", "candidate_id"]).reset_index(drop=True)
    return frame


def stable_project_seed(project, scope_name):
    token = hashlib.sha256(f"{RANDOM_SEED}|{project}|{scope_name}".encode()).hexdigest()
    return int(token[:8], 16)


def build_ce_pairs_for_project(data, allowed_ids, scope_name, dense_model, dense_fingerprint):
    allowed = set(allowed_ids)
    positives = build_positive_pairs(data, allowed)
    assert len(positives), f"{data['project']}: no positive pairs in {scope_name}"
    anchor_queries = positives[["query_id", "query_time"]].drop_duplicates().reset_index(drop=True)
    print(f"{data['project']}: {len(anchor_queries):,} positive query groups")

    embeddings = encode_dense_project(
        dense_model, data, f"{scope_name}_hard_negative_miner", dense_fingerprint
    )
    mined = retrieve_dense_rolling(
        data, anchor_queries, embeddings, allowed, HARD_NEGATIVE_SEARCH_DEPTH
    )
    ranking_map = mined.set_index("query_id")["retrieved_ids"].map(split_ids).to_dict()
    print(f"{data['project']}: dense mining complete; building compact ID pairs...")

    components = known_duplicate_components(data, allowed)
    issues = data["issues"]
    rows = []
    positive_map = positives.groupby("query_id")["candidate_id"].apply(list).to_dict()

    # Random candidates require a corpus scan per query. Skip that code path entirely
    # when the configured count is zero; the previous implementation performed this
    # expensive scan anyway and appeared to hang after dense mining.
    random_enabled = RANDOM_NEGATIVES_PER_QUERY > 0
    rng = (
        np.random.default_rng(stable_project_seed(data["project"], scope_name))
        if random_enabled else None
    )
    if not random_enabled:
        print(f"{data['project']}: random-negative scan skipped (configured count = 0)")

    for number, query_id in enumerate(anchor_queries["query_id"], start=1):
        blocked = components[query_id]
        for candidate_id in positive_map[query_id]:
            rows.append({
                "project": data["project"], "scope": scope_name,
                "query_id": query_id, "candidate_id": candidate_id,
                "label": 1.0, "negative_kind": "positive",
            })

        hard = [x for x in ranking_map[query_id] if x not in blocked]
        hard = hard[:HARD_NEGATIVES_PER_QUERY]
        for candidate_id in hard:
            rows.append({
                "project": data["project"], "scope": scope_name,
                "query_id": query_id, "candidate_id": candidate_id,
                "label": 0.0, "negative_kind": "hard_dense",
            })

        if random_enabled:
            query_time = data["id_to_time"][query_id]
            legal_random = issues.loc[
                issues["issue_id"].isin(allowed)
                & (issues["created_at"] < query_time)
                & issues["ce_text"].str.len().gt(0), "issue_id"
            ].tolist()
            legal_random = [x for x in legal_random if x not in blocked and x not in hard]
            take = min(RANDOM_NEGATIVES_PER_QUERY, len(legal_random))
            if take:
                sampled = rng.choice(legal_random, size=take, replace=False).tolist()
                for candidate_id in sampled:
                    rows.append({
                        "project": data["project"], "scope": scope_name,
                        "query_id": query_id, "candidate_id": candidate_id,
                        "label": 0.0, "negative_kind": "random",
                    })

        if number % PAIR_ASSEMBLY_PROGRESS_EVERY == 0 or number == len(anchor_queries):
            print(f"  {data['project']} pair assembly: {number:,}/{len(anchor_queries):,}")

    frame = pd.DataFrame(rows).drop_duplicates(
        ["project", "query_id", "candidate_id", "label"]
    ).reset_index(drop=True)

    # O(number_of_pairs), without pandas apply(axis=1) and per-row Series creation.
    negative_pairs = frame.loc[
        frame["label"].eq(0), ["query_id", "candidate_id"]
    ].itertuples(index=False, name=None)
    has_known_duplicate_negative = any(
        candidate_id in components[query_id]
        for query_id, candidate_id in negative_pairs
    )
    assert not has_known_duplicate_negative
    assert set(frame["query_id"]).issubset(allowed)
    assert set(frame["candidate_id"]).issubset(allowed)
    assert all(
        data["id_to_time"][candidate_id] < data["id_to_time"][query_id]
        for query_id, candidate_id in frame[["query_id", "candidate_id"]].itertuples(
            index=False, name=None
        )
    )
    print(f"{data['project']}: compact pair frame ready — {len(frame):,} rows")
    return frame


def save_pair_dataset(frame, stem, metadata):
    csv_path = OUTPUT_ROOT / f"{stem}.csv"
    json_path = OUTPUT_ROOT / f"{stem}.json"
    print(f"Writing compact pair IDs: {csv_path.name} ({len(frame):,} rows)")
    frame.to_csv(csv_path, index=False)
    json_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    print("PAIR ARTIFACT WRITTEN:", csv_path)
    return csv_path


def pair_metadata(scope, project, dense_fingerprint, rows=None):
    result = {
        "scope": scope, "project": project,
        "pair_schema": "ids_only_v2",
        "used_validation_labels": scope == "official_train",
        "used_official_test_labels": False,
        "dense_miner_checkpoint_fingerprint": dense_fingerprint,
        "hard_negative_search_depth": HARD_NEGATIVE_SEARCH_DEPTH,
        "hard_negatives_per_query": HARD_NEGATIVES_PER_QUERY,
        "random_negatives_per_query": RANDOM_NEGATIVES_PER_QUERY,
        "max_positives_per_query": MAX_POSITIVES_PER_QUERY,
        "known_component_exclusion": True,
    }
    if rows is not None:
        result["rows"] = int(rows)
    return result


def metadata_matches(path, expected):
    if not path.exists():
        return False
    observed = json.loads(path.read_text(encoding="utf-8"))
    return all(observed.get(key) == value for key, value in expected.items())


In [ ]:
dev_training_pairs = pd.DataFrame()

if RUN_BUILD_DEV_TRAINING_DATA:
    dev_dense_fingerprint = checkpoint_fingerprint(DEV_DENSE_CHECKPOINT)
    dev_dense_model = None
    frames = []
    for project, data in datasets.items():
        print(f"\n=== BUILD DEV PAIRS: {project} ===")
        stem = f"{project.lower()}_dev_fit_training_pairs"
        csv_path = OUTPUT_ROOT / f"{stem}.csv"
        json_path = OUTPUT_ROOT / f"{stem}.json"
        expected = pair_metadata("dev_fit", project, dev_dense_fingerprint)
        if (
            csv_path.exists() and metadata_matches(json_path, expected)
            and not FORCE_REBUILD_PAIR_DATA
        ):
            print("PAIR CACHE HIT:", csv_path)
            frame = pd.read_csv(
                csv_path, dtype={"query_id": str, "candidate_id": str}
            )
        else:
            if dev_dense_model is None:
                dev_dense_model = SentenceTransformer(str(DEV_DENSE_CHECKPOINT))
            frame = build_ce_pairs_for_project(
                data, query_sets[project]["dev_fit_ids"], "dev_fit",
                dev_dense_model, dev_dense_fingerprint,
            )
            save_pair_dataset(
                frame, stem,
                pair_metadata("dev_fit", project, dev_dense_fingerprint, len(frame)),
            )
        frames.append(frame)

    dev_training_pairs = pd.concat(frames, ignore_index=True)
    dev_pair_audit = (
        dev_training_pairs.groupby(["project", "negative_kind"])
        .size().rename("pairs").reset_index()
    )
    display(dev_pair_audit)
    metadata = pair_metadata(
        "dev_fit", "ALL_PROJECTS", dev_dense_fingerprint, len(dev_training_pairs)
    )
    metadata["used_validation_labels"] = False
    save_pair_dataset(dev_training_pairs, "dev_training_pairs", metadata)
    if dev_dense_model is not None:
        del dev_dense_model
    del frames
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
else:
    print("DEV PAIR BUILD LOCKED.")


## 6. Fine-tune with grouped pairwise RankNet loss

Each training item is one query group containing one historical positive and up to
three dense hard negatives. The loss directly optimizes every valid comparison:

```text
softplus(-sigma * (score(query, positive) - score(query, hard_negative)))
```

Thus a positive must outrank the negatives for the **same query**. No global class
threshold and no `pos_weight` are used. BCE was not intrinsically invalid—it is a
strong standard baseline—but the observed ranking regression justifies this closer
alignment with the Top-50 reranking objective. Selection remains end-to-end temporal
validation retrieval, never pair-classification accuracy.


In [ ]:
class GroupedQueryDataset(Dataset):
    """One item = one query with candidate IDs; text is resolved lazily."""

    def __init__(self, frame, project_datasets):
        self.groups = []
        self.text_lookup = {
            project: data["id_to_ce_text"]
            for project, data in project_datasets.items()
        }
        group_columns = ["project", "query_id"]
        for (project, query_id), group in frame.groupby(group_columns, sort=True):
            positives = group[group["label"] == 1.0]
            negatives = group[group["label"] == 0.0]
            if len(positives) == 0 or len(negatives) == 0:
                continue
            candidate_ids = (
                positives["candidate_id"].tolist()
                + negatives["candidate_id"].tolist()
            )
            labels = [1.0] * len(positives) + [0.0] * len(negatives)
            assert self.text_lookup[project].get(query_id, "")
            assert all(self.text_lookup[project].get(x, "") for x in candidate_ids)
            self.groups.append({
                "project": project, "query_id": query_id,
                "candidate_ids": candidate_ids, "labels": labels,
                "positive_count": len(positives),
                "negative_count": len(negatives),
            })
        assert self.groups, "No valid query groups with both positive and negative documents"

    def __len__(self):
        return len(self.groups)

    def __getitem__(self, index):
        group = self.groups[index]
        lookup = self.text_lookup[group["project"]]
        return {
            **group,
            "query_text": lookup[group["query_id"]],
            "documents": [lookup[x] for x in group["candidate_ids"]],
        }

class GroupedQueryCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, batch):
        query_texts, document_texts, labels, group_sizes = [], [], [], []
        for group in batch:
            size = len(group["documents"])
            group_sizes.append(size)
            query_texts.extend([group["query_text"]] * size)
            document_texts.extend(group["documents"])
            labels.extend(group["labels"])
        encoded = self.tokenizer(
            query_texts, document_texts, padding=True, truncation=True,
            max_length=CE_MAX_LENGTH, return_tensors="pt",
        )
        encoded["labels"] = torch.tensor(labels, dtype=torch.float32)
        encoded["group_sizes"] = group_sizes
        return encoded


def pairwise_ranknet_loss(scores, labels, group_sizes, sigma=PAIRWISE_SIGMA):
    """Mean RankNet logistic loss across queries, then across pos-neg pairs."""
    cursor, group_losses = 0, []
    for size in group_sizes:
        group_scores = scores[cursor:cursor + size]
        group_labels = labels[cursor:cursor + size]
        cursor += size
        positive_scores = group_scores[group_labels > 0.5]
        negative_scores = group_scores[group_labels < 0.5]
        assert positive_scores.numel() > 0 and negative_scores.numel() > 0
        differences = positive_scores[:, None] - negative_scores[None, :]
        group_losses.append(torch.nn.functional.softplus(-sigma * differences).mean())
    assert cursor == len(scores) and group_losses
    return torch.stack(group_losses).mean()


def base_ce_source():
    if CE_BASE_LOCAL_PATH:
        path = Path(CE_BASE_LOCAL_PATH)
        assert path.exists(), path
        return str(path), {}
    return CE_BASE_MODEL, {"revision": CE_BASE_REVISION}


def train_cross_encoder(pair_frame, output_path, scope_name, dense_miner_fingerprint):
    pair_frame = pair_frame.copy()
    pair_frame["label"] = pair_frame["label"].astype(float)
    assert set(pair_frame["label"].unique()) == {0.0, 1.0}
    assert len(pair_frame) == len(pair_frame.drop_duplicates(
        ["project", "query_id", "candidate_id", "label"]
    ))
    grouped_dataset = GroupedQueryDataset(pair_frame, datasets)
    model_source, revision_kwargs = base_ce_source()
    tokenizer = AutoTokenizer.from_pretrained(
        model_source, trust_remote_code=False, **revision_kwargs
    )
    model = AutoModelForSequenceClassification.from_pretrained(
        model_source, num_labels=1, trust_remote_code=False, **revision_kwargs
    )
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    generator = torch.Generator().manual_seed(RANDOM_SEED)
    loader = DataLoader(
        grouped_dataset, batch_size=CE_QUERY_BATCH_SIZE,
        shuffle=True, generator=generator, num_workers=0,
        collate_fn=GroupedQueryCollator(tokenizer),
        pin_memory=torch.cuda.is_available(),
    )
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=CE_LEARNING_RATE, weight_decay=CE_WEIGHT_DECAY
    )
    update_steps_per_epoch = math.ceil(len(loader) / CE_GRADIENT_ACCUMULATION)
    total_steps = update_steps_per_epoch * CE_EPOCHS
    warmup_steps = int(total_steps * CE_WARMUP_RATIO)
    scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
    use_amp = torch.cuda.is_available()
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
    history = []
    optimizer.zero_grad(set_to_none=True)
    for epoch in range(CE_EPOCHS):
        model.train()
        running_loss, seen_groups = 0.0, 0
        for step, batch in enumerate(loader, start=1):
            labels = batch.pop("labels").to(device)
            group_sizes = batch.pop("group_sizes")
            batch = {key: value.to(device) for key, value in batch.items()}
            with torch.cuda.amp.autocast(enabled=use_amp):
                logits = model(**batch).logits.squeeze(-1)
                raw_loss = pairwise_ranknet_loss(logits, labels, group_sizes)
                loss = raw_loss / CE_GRADIENT_ACCUMULATION
            scaler.scale(loss).backward()
            if step % CE_GRADIENT_ACCUMULATION == 0 or step == len(loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
            batch_groups = len(group_sizes)
            running_loss += float(raw_loss.detach().cpu()) * batch_groups
            seen_groups += batch_groups
            if step % 250 == 0 or step == len(loader):
                print(f"epoch {epoch + 1}/{CE_EPOCHS} step {step}/{len(loader)}")
        epoch_loss = running_loss / seen_groups
        history.append({"epoch": epoch + 1, "train_pairwise_ranknet": epoch_loss})
        print("epoch pairwise RankNet loss:", round(epoch_loss, 6))

    output_path = Path(output_path)
    output_path.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(output_path, safe_serialization=True)
    tokenizer.save_pretrained(output_path)
    positives = int((pair_frame["label"] == 1).sum())
    negatives = int((pair_frame["label"] == 0).sum())
    comparisons = sum(
        group["positive_count"] * group["negative_count"]
        for group in grouped_dataset.groups
    )
    manifest = {
        "phase": "5_v2", "architecture": "cross_encoder",
        "objective": "grouped_pairwise_ranknet",
        "scope": scope_name,
        "base_model": CE_BASE_MODEL, "base_revision": CE_BASE_REVISION,
        "started_from_dev_checkpoint": False,
        "used_validation_labels": False if scope_name == "dev_fit" else True,
        "used_official_test_pairs": False,
        "dense_miner_fingerprint": dense_miner_fingerprint,
        "query_groups": len(grouped_dataset),
        "flat_pairs": len(pair_frame), "positives": positives, "negatives": negatives,
        "pairwise_comparisons": comparisons,
        "pairwise_sigma": PAIRWISE_SIGMA,
        "epochs": CE_EPOCHS, "query_batch_size": CE_QUERY_BATCH_SIZE,
        "gradient_accumulation": CE_GRADIENT_ACCUMULATION,
        "learning_rate": CE_LEARNING_RATE, "max_length": CE_MAX_LENGTH,
        "random_negatives_per_query": RANDOM_NEGATIVES_PER_QUERY,
        "history": history,
    }
    (output_path.parent / "training_manifest.json").write_text(
        json.dumps(manifest, indent=2), encoding="utf-8"
    )
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return manifest


# The desired ordering must have lower loss than the reversed ordering.
_labels = torch.tensor([1.0, 0.0, 0.0])
_good = pairwise_ranknet_loss(torch.tensor([2.0, 0.0, -1.0]), _labels, [3])
_bad = pairwise_ranknet_loss(torch.tensor([-1.0, 1.0, 2.0]), _labels, [3])
assert _good < _bad
print("PAIRWISE RANKNET LOSS TEST: PASS")


if RUN_TRAIN_DEV_CROSS_ENCODER:
    pair_path = phase5_artifact_for_read("dev_training_pairs.csv")
    meta_path = phase5_artifact_for_read("dev_training_pairs.json")
    assert pair_path.exists() and meta_path.exists()
    dev_training_pairs = pd.read_csv(pair_path)
    pair_meta = json.loads(meta_path.read_text(encoding="utf-8"))
    assert pair_meta["scope"] == "dev_fit"
    assert pair_meta["used_validation_labels"] is False
    assert pair_meta["used_official_test_labels"] is False
    assert pair_meta["random_negatives_per_query"] == RANDOM_NEGATIVES_PER_QUERY
    assert pair_meta["pair_schema"] == "ids_only_v2"
    dev_training_manifest = train_cross_encoder(
        dev_training_pairs, DEV_CE_OUTPUT, "dev_fit",
        pair_meta["dense_miner_checkpoint_fingerprint"],
    )
    print(json.dumps(dev_training_manifest, indent=2))
else:
    print("DEV CROSS-ENCODER TRAINING LOCKED.")


## 7. End-to-end validation reranking

The candidate set is unchanged: cross-encoder scores only reorder the dense
Top-50. The table reports dense and reranked Hit/Recall@1/5/10, MRR@10, nDCG@10,
candidate Hit@50, and paired wins/losses. Candidate Hit@50 is the maximum Hit@10
any reranker can attain for this candidate generator.

In [ ]:
def load_ce_checkpoint(checkpoint, revision=None):
    checkpoint_text = str(checkpoint)
    checkpoint_path = Path(checkpoint_text)
    is_local = checkpoint_path.exists()
    kwargs = {} if is_local else {"revision": revision}
    tokenizer = AutoTokenizer.from_pretrained(
        checkpoint_text, trust_remote_code=False, **kwargs
    )
    model = AutoModelForSequenceClassification.from_pretrained(
        checkpoint_text, trust_remote_code=False, **kwargs
    )
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device).eval()
    return tokenizer, model, device


def score_cross_encoder_pairs(tokenizer, model, device, pairs):
    scores = []
    with torch.inference_mode():
        for start in range(0, len(pairs), CE_INFERENCE_BATCH_SIZE):
            batch_pairs = pairs[start:start + CE_INFERENCE_BATCH_SIZE]
            encoded = tokenizer(
                [x[0] for x in batch_pairs], [x[1] for x in batch_pairs],
                padding=True, truncation=True, max_length=CE_MAX_LENGTH,
                return_tensors="pt",
            )
            encoded = {key: value.to(device) for key, value in encoded.items()}
            logits = model(**encoded).logits.squeeze(-1).float().cpu().numpy()
            scores.extend(np.atleast_1d(logits).tolist())
    return np.asarray(scores, dtype=np.float32)


def rerank_dense_frame(data, queries, dense_frame, ce_checkpoint, split, ce_revision=None):
    tokenizer, model, device = load_ce_checkpoint(ce_checkpoint, ce_revision)
    relevant_map = queries.set_index("query_id")["relevant_ids"].to_dict()
    rows = []
    for number, row in enumerate(dense_frame.itertuples(index=False), start=1):
        candidates = split_ids(row.retrieved_ids)
        query_text = data["id_to_ce_text"][row.query_id]
        pairs = [(query_text, data["id_to_ce_text"][x]) for x in candidates]
        scores = score_cross_encoder_pairs(tokenizer, model, device, pairs)
        order = np.lexsort((np.arange(len(candidates)), -scores))
        reranked = [candidates[i] for i in order]
        sorted_scores = [float(scores[i]) for i in order]
        assert set(reranked) == set(candidates)
        relevant = relevant_map[row.query_id]
        rows.append({
            "project": data["project"], "split": split,
            "query_id": row.query_id, "query_time": row.query_time,
            "dense_ids": ";".join(candidates),
            "reranked_ids": ";".join(reranked),
            "reranked_scores": ";".join(f"{x:.7g}" for x in sorted_scores),
            "candidate_hit@50": float(bool(set(candidates) & set(relevant))),
            **{f"dense_{k}": v for k, v in ranking_metrics(candidates, relevant).items()},
            **{f"reranked_{k}": v for k, v in ranking_metrics(reranked, relevant).items()},
        })
        if number % 250 == 0 or number == len(dense_frame):
            print(f"  {data['project']} cross-encoder: {number}/{len(dense_frame)}")
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return pd.DataFrame(rows)


def summarize_reranking(frame):
    row = {
        "project": frame["project"].iloc[0], "queries": len(frame),
        "candidate_hit@50": frame["candidate_hit@50"].mean(),
    }
    metric_names = [
        *(f"hit@{k}" for k in METRIC_KS),
        *(f"recall@{k}" for k in METRIC_KS), "mrr@10", "ndcg@10",
    ]
    for system in ("dense", "reranked"):
        for metric in metric_names:
            row[f"{system}_{metric}"] = frame[f"{system}_{metric}"].mean()
    return row


def macro_reranking_row(project_summary):
    numeric = [
        x for x in project_summary.columns
        if x not in {"project", "queries"}
    ]
    return {
        "project": "MACRO_PROJECT_AVERAGE",
        "queries": int(project_summary["queries"].sum()),
        **{column: project_summary[column].mean() for column in numeric},
    }


def selection_tuple_from_summary(row, prefix):
    return tuple(float(row[f"{prefix}_{metric}"]) for metric in (
        PRIMARY_SELECTION_METRIC, *SELECTION_TIE_BREAKERS
    ))


In [ ]:
validation_reranked = {}
validation_summary = pd.DataFrame()
pretrained_validation_summary = pd.DataFrame()

if RUN_DEV_VALIDATION_RERANK:
    dev_ce_checkpoint = phase5_artifact_for_read("dev_cross_encoder/final")
    assert dev_ce_checkpoint.exists(), dev_ce_checkpoint
    dev_manifest_path = dev_ce_checkpoint.parent / "training_manifest.json"
    assert dev_manifest_path.exists()
    dev_manifest = json.loads(dev_manifest_path.read_text(encoding="utf-8"))
    assert dev_manifest["scope"] == "dev_fit"
    assert dev_manifest["objective"] == "grouped_pairwise_ranknet"
    assert dev_manifest["used_validation_labels"] is False
    assert dev_manifest["used_official_test_pairs"] is False

    dev_dense_fingerprint = checkpoint_fingerprint(DEV_DENSE_CHECKPOINT)
    dev_dense_model = SentenceTransformer(str(DEV_DENSE_CHECKPOINT))
    summaries, pretrained_summaries = [], []
    for project, data in datasets.items():
        print(f"\n=== DEV VALIDATION RERANK: {project} ===")
        queries = query_sets[project]["validation"]
        ranking_path = OUTPUT_ROOT / f"{project.lower()}_validation_dense_top50.csv"
        if ranking_path.exists() and not FORCE_REBUILD_RANKINGS:
            dense_frame = pd.read_csv(ranking_path, dtype={"query_id": str, "retrieved_ids": str})
        else:
            embeddings = encode_dense_project(
                dev_dense_model, data, "dev_validation", dev_dense_fingerprint
            )
            dense_frame = retrieve_dense_rolling(
                data, queries, embeddings, data["official_train_ids"], CANDIDATE_DEPTH
            )
            dense_frame.to_csv(ranking_path, index=False)
        dense_frame = validate_ranking_frame(
            dense_frame, data, queries, data["official_train_ids"], CANDIDATE_DEPTH
        )
        reranked = rerank_dense_frame(
            data, queries, dense_frame, dev_ce_checkpoint, "validation"
        )
        validation_reranked[project] = reranked
        reranked.to_csv(
            OUTPUT_ROOT / f"{project.lower()}_validation_cross_encoder_queries.csv",
            index=False,
        )
        summaries.append(summarize_reranking(reranked))
        if RUN_PRETRAINED_CE_VALIDATION:
            base_source = CE_BASE_LOCAL_PATH or CE_BASE_MODEL
            base_revision = None if CE_BASE_LOCAL_PATH else CE_BASE_REVISION
            base_reranked = rerank_dense_frame(
                data, queries, dense_frame, base_source,
                "validation_pretrained_ce", base_revision,
            )
            base_reranked.to_csv(
                OUTPUT_ROOT / f"{project.lower()}_validation_pretrained_ce_queries.csv",
                index=False,
            )
            pretrained_summaries.append(summarize_reranking(base_reranked))
    project_summary = pd.DataFrame(summaries)
    validation_summary = pd.concat([
        project_summary, pd.DataFrame([macro_reranking_row(project_summary)])
    ], ignore_index=True)
    display(validation_summary)
    validation_summary.to_csv(OUTPUT_ROOT / "validation_reranking_summary.csv", index=False)
    if pretrained_summaries:
        base_project_summary = pd.DataFrame(pretrained_summaries)
        pretrained_validation_summary = pd.concat([
            base_project_summary,
            pd.DataFrame([macro_reranking_row(base_project_summary)]),
        ], ignore_index=True)
        print("Pretrained cross-encoder baseline:")
        display(pretrained_validation_summary)
        pretrained_validation_summary.to_csv(
            OUTPUT_ROOT / "pretrained_ce_validation_summary.csv", index=False
        )

    paired_rows = []
    for project, frame in validation_reranked.items():
        temp = frame[["query_id", "dense_hit@10", "reranked_hit@10"]].copy()
        temp["project"] = project
        temp["outcome"] = np.select(
            [
                (temp["reranked_hit@10"] == 1) & (temp["dense_hit@10"] == 0),
                (temp["reranked_hit@10"] == 0) & (temp["dense_hit@10"] == 1),
                (temp["reranked_hit@10"] == 1) & (temp["dense_hit@10"] == 1),
            ], ["reranker_win", "dense_win", "both_correct"], default="both_fail",
        )
        paired_rows.append(temp)
    paired = pd.concat(paired_rows, ignore_index=True)
    paired_summary = paired.groupby(["project", "outcome"]).size().rename("queries").reset_index()
    display(paired_summary)
    paired.to_csv(OUTPUT_ROOT / "validation_paired_queries.csv", index=False)
    paired_summary.to_csv(OUTPUT_ROOT / "validation_paired_summary.csv", index=False)
    del dev_dense_model
else:
    print("DEV VALIDATION RERANK LOCKED.")


## 8. Freeze validation decision

The fine-tuned cross-encoder must beat dense retrieval on macro-project Hit@10;
MRR@10 and Recall@10 are frozen tie-breakers. One fixed training configuration is
used—there is no hidden hyperparameter sweep. The full validation query count and
candidate coverage are persisted before any Phase 5 test run.

In [ ]:
phase5_selection = None
selection_path = OUTPUT_ROOT / "phase5_selection.json"

if RUN_DEV_VALIDATION_RERANK:
    macro = validation_summary.query("project == 'MACRO_PROJECT_AVERAGE'").iloc[0]
    dense_score = selection_tuple_from_summary(macro, "dense")
    reranker_score = selection_tuple_from_summary(macro, "reranked")
    reranker_wins = reranker_score > dense_score
    pretrained_ce_score = None
    if len(pretrained_validation_summary):
        pretrained_macro = pretrained_validation_summary.query(
            "project == 'MACRO_PROJECT_AVERAGE'"
        ).iloc[0]
        pretrained_ce_score = list(selection_tuple_from_summary(pretrained_macro, "reranked"))
    phase5_selection = {
        "phase": "5_v2",
        "training_objective": "grouped_pairwise_ranknet",
        "candidate_generator": "phase3_fine_tuned_minilm",
        "candidate_depth": CANDIDATE_DEPTH, "output_k": OUTPUT_K,
        "aggregation": SELECTION_AGGREGATION,
        "primary_metric": PRIMARY_SELECTION_METRIC,
        "tie_breakers": list(SELECTION_TIE_BREAKERS),
        "dense_score_tuple": list(dense_score),
        "pretrained_cross_encoder_score_tuple": pretrained_ce_score,
        "reranker_score_tuple": list(reranker_score),
        "reranker_beats_dense": reranker_wins,
        "winner": "fine_tuned_cross_encoder" if reranker_wins else "fine_tuned_dense",
        "validation_candidate_hit_at_50": float(macro["candidate_hit@50"]),
        "full_validation": all(
            len(validation_reranked[p]) == len(query_sets[p]["validation"])
            for p in PROJECTS
        ),
        "used_validation_pairs_for_training": False,
        "used_official_test_pairs_for_training_or_selection": False,
        "official_test_was_previously_opened_in_phase4": True,
        "official_test_claim_status": "exploratory_reused_holdout",
        "final_retrain_allowed": reranker_wins,
    }
    selection_path.write_text(json.dumps(phase5_selection, indent=2), encoding="utf-8")
    print(json.dumps(phase5_selection, indent=2))
else:
    print("SELECTION WAITS FOR COMPLETE VALIDATION.")


## 9. Final retrain on full official train

Only if validation selected the reranker, rebuild positives and negatives using
full official train and the Phase 3 final dense checkpoint. Restart from the
original pinned cross-encoder base—not the development cross-encoder—so the final
model has a clean, auditable training scope.

In [ ]:
final_training_pairs = pd.DataFrame()

if RUN_BUILD_FINAL_TRAINING_DATA:
    selection_read_path = phase5_artifact_for_read("phase5_selection.json")
    assert selection_read_path.exists(), selection_read_path
    selected = json.loads(selection_read_path.read_text(encoding="utf-8"))
    assert selected["reranker_beats_dense"] is True
    assert selected["full_validation"] is True
    assert selected["used_official_test_pairs_for_training_or_selection"] is False
    final_dense_fingerprint = checkpoint_fingerprint(FINAL_DENSE_CHECKPOINT)
    final_dense_model = SentenceTransformer(str(FINAL_DENSE_CHECKPOINT))
    frames = []
    for project, data in datasets.items():
        print(f"\n=== BUILD FINAL TRAIN PAIRS: {project} ===")
        frame = build_ce_pairs_for_project(
            data, data["official_train_ids"], "official_train",
            final_dense_model, final_dense_fingerprint,
        )
        frames.append(frame)
    final_training_pairs = pd.concat(frames, ignore_index=True)
    display(
        final_training_pairs.groupby(["project", "negative_kind"])
        .size().rename("pairs").reset_index()
    )
    metadata = pair_metadata(
        "official_train", "ALL_PROJECTS", final_dense_fingerprint,
        len(final_training_pairs),
    )
    metadata["training_configuration_frozen_from_dev"] = True
    save_pair_dataset(final_training_pairs, "final_training_pairs", metadata)
    del final_dense_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
else:
    print("FINAL PAIR BUILD LOCKED.")


if RUN_FINAL_RETRAIN:
    selection_read_path = phase5_artifact_for_read("phase5_selection.json")
    assert selection_read_path.exists()
    selected = json.loads(selection_read_path.read_text(encoding="utf-8"))
    assert selected["final_retrain_allowed"] is True
    pair_path = phase5_artifact_for_read("final_training_pairs.csv")
    meta_path = phase5_artifact_for_read("final_training_pairs.json")
    assert pair_path.exists() and meta_path.exists()
    final_training_pairs = pd.read_csv(pair_path)
    pair_meta = json.loads(meta_path.read_text(encoding="utf-8"))
    assert pair_meta["scope"] == "official_train"
    assert pair_meta["used_official_test_labels"] is False
    assert pair_meta["pair_schema"] == "ids_only_v2"
    assert pair_meta["training_configuration_frozen_from_dev"] is True
    final_training_manifest = train_cross_encoder(
        final_training_pairs, FINAL_CE_OUTPUT, "official_train",
        pair_meta["dense_miner_checkpoint_fingerprint"],
    )
    assert final_training_manifest["started_from_dev_checkpoint"] is False
    assert final_training_manifest["objective"] == "grouped_pairwise_ranknet"
    assert final_training_manifest["used_official_test_pairs"] is False
    print(json.dumps(final_training_manifest, indent=2))
else:
    print("FINAL CROSS-ENCODER RETRAIN LOCKED.")


## 10. Reused official test — exploratory only

Phase 4 already displayed official-test aggregate metrics (including hybrid
Hit@10 around 63.10%). Phase 5 must not pretend this is an untouched test. To run
the comparison anyway, set both acknowledgement and run switches to `True`.

The notebook reuses Phase 4 final dense Top-50 CSVs when present; otherwise it
recomputes them with the frozen Phase 3 final checkpoint. It never uses test
labels for training or model selection.

In [ ]:
def phase4_dense_ranking_path(project):
    name = f"{project.lower()}_official_test_dense_top50.csv"
    candidates = [
        PHASE4_INPUT_ROOT / name,
        Path("/kaggle/working/phase4_rrf_outputs") / name,
    ]
    return next((path for path in candidates if path.exists()), None)


def existing_final_ce_checkpoint():
    candidates = [
        FINAL_CE_OUTPUT,
        PHASE5_INPUT_ROOT / "final_cross_encoder" / "final",
    ]
    return next((path for path in candidates if path.exists()), None)


official_test_summary = pd.DataFrame()

if RUN_OFFICIAL_TEST:
    assert OFFICIAL_TEST_WAS_PREVIOUSLY_OPENED_IN_PHASE4 is True
    assert ALLOW_REUSED_OFFICIAL_TEST_FOR_EXPLORATORY_REPORT is True, (
        "This is a reused holdout. Set the acknowledgement switch explicitly."
    )
    selection_read_path = phase5_artifact_for_read("phase5_selection.json")
    assert selection_read_path.exists(), selection_read_path
    selected = json.loads(selection_read_path.read_text(encoding="utf-8"))
    assert selected["winner"] == "fine_tuned_cross_encoder"
    assert selected["full_validation"] is True
    assert selected["used_official_test_pairs_for_training_or_selection"] is False
    final_ce_checkpoint = existing_final_ce_checkpoint()
    assert final_ce_checkpoint is not None, "Final cross-encoder checkpoint missing"
    final_manifest_path = final_ce_checkpoint.parent / "training_manifest.json"
    assert final_manifest_path.exists(), final_manifest_path
    final_manifest = json.loads(final_manifest_path.read_text(encoding="utf-8"))
    assert final_manifest["scope"] == "official_train"
    assert final_manifest["objective"] == "grouped_pairwise_ranknet"
    assert final_manifest["started_from_dev_checkpoint"] is False
    assert final_manifest["used_official_test_pairs"] is False

    final_dense_fingerprint = checkpoint_fingerprint(FINAL_DENSE_CHECKPOINT)
    final_dense_model = None
    summaries, test_reranked = [], {}
    for project, data in datasets.items():
        print(f"\n=== EXPLORATORY REUSED TEST: {project} ===")
        queries = query_sets[project]["test"]
        allowed = data["official_train_ids"] | data["official_test_ids"]
        source_path = phase4_dense_ranking_path(project)
        if source_path is not None:
            print("Reusing Phase 4 dense candidates:", source_path)
            dense_frame = pd.read_csv(
                source_path, dtype={"query_id": str, "retrieved_ids": str}
            )
        else:
            print("Phase 4 artifact missing; recomputing frozen dense candidates.")
            if final_dense_model is None:
                final_dense_model = SentenceTransformer(str(FINAL_DENSE_CHECKPOINT))
            embeddings = encode_dense_project(
                final_dense_model, data, "final_test", final_dense_fingerprint
            )
            dense_frame = retrieve_dense_rolling(
                data, queries, embeddings, allowed, CANDIDATE_DEPTH
            )
        dense_frame = validate_ranking_frame(
            dense_frame, data, queries, allowed, CANDIDATE_DEPTH
        )
        reranked = rerank_dense_frame(
            data, queries, dense_frame, final_ce_checkpoint, "official_test_reused"
        )
        test_reranked[project] = reranked
        reranked.to_csv(
            OUTPUT_ROOT / f"{project.lower()}_reused_test_cross_encoder_queries.csv",
            index=False,
        )
        summaries.append(summarize_reranking(reranked))
    project_summary = pd.DataFrame(summaries)
    official_test_summary = pd.concat([
        project_summary, pd.DataFrame([macro_reranking_row(project_summary)])
    ], ignore_index=True)
    official_test_summary["claim_status"] = "EXPLORATORY_REUSED_HOLDOUT"
    display(official_test_summary)
    official_test_summary.to_csv(
        OUTPUT_ROOT / "reused_official_test_reranking_summary.csv", index=False
    )

    # Optional side-by-side display with the Phase 4 summary; never a selection gate.
    p4_summary_candidates = [
        PHASE4_INPUT_ROOT / "official_test_summary.csv",
        Path("/kaggle/working/phase4_rrf_outputs/official_test_summary.csv"),
    ]
    p4_summary_path = next((p for p in p4_summary_candidates if p.exists()), None)
    if p4_summary_path:
        print("Phase 4 comparison (descriptive only):")
        display(pd.read_csv(p4_summary_path))
    if final_dense_model is not None:
        del final_dense_model
else:
    print("OFFICIAL TEST SEALED. This test was already opened in Phase 4.")


## 11. Manifest and final leakage assertions

In [ ]:
run_manifest = {
    "phase": "5_v2",
    "method": "fine_tuned_cross_encoder_pairwise_ranknet_reranking",
    "source_notebook": "phase4-rrf-hybrid-benchmark-ipynb.ipynb",
    "candidate_generator": "phase3_fine_tuned_minilm",
    "candidate_depth": CANDIDATE_DEPTH,
    "output_k": OUTPUT_K,
    "cross_encoder": {
        "base_model": CE_BASE_MODEL, "base_revision": CE_BASE_REVISION,
        "max_length": CE_MAX_LENGTH, "epochs": CE_EPOCHS,
        "objective": "grouped_pairwise_ranknet",
        "pairwise_sigma": PAIRWISE_SIGMA,
        "query_batch_size": CE_QUERY_BATCH_SIZE,
        "learning_rate": CE_LEARNING_RATE,
        "hard_negatives_per_query": HARD_NEGATIVES_PER_QUERY,
        "random_negatives_per_query": RANDOM_NEGATIVES_PER_QUERY,
        "known_duplicate_component_exclusion": True,
        "pair_schema": "ids_only_v2",
        "random_scan_skipped_when_zero": True,
        "per_project_pair_cache": True,
    },
    "selection_rule": {
        "aggregation": SELECTION_AGGREGATION,
        "primary_metric": PRIMARY_SELECTION_METRIC,
        "tie_breakers": list(SELECTION_TIE_BREAKERS),
    },
    "switches": {
        "build_dev_pairs": RUN_BUILD_DEV_TRAINING_DATA,
        "train_dev": RUN_TRAIN_DEV_CROSS_ENCODER,
        "pretrained_ce_validation": RUN_PRETRAINED_CE_VALIDATION,
        "validate": RUN_DEV_VALIDATION_RERANK,
        "build_final_pairs": RUN_BUILD_FINAL_TRAINING_DATA,
        "final_retrain": RUN_FINAL_RETRAIN,
        "official_test": RUN_OFFICIAL_TEST,
    },
    "leakage_contract": {
        "dev_training_scope": "dev_fit",
        "validation_labels_used_for_dev_training": False,
        "official_test_labels_used_for_training_or_selection": False,
        "candidate_timestamp_policy": "strictly_older_than_query",
        "final_model_restart_from_pinned_base": True,
        "official_test_previously_opened_in_phase4": True,
        "official_test_claim_status": "exploratory_reused_holdout",
    },
    "environment": {
        "python": sys.version.split()[0], "platform": platform.platform(),
        "torch": torch.__version__, "transformers": transformers.__version__,
        "sentence_transformers": sentence_transformers.__version__,
        "cuda_available": torch.cuda.is_available(),
    },
    "projects": {
        project: {
            "issues": len(data["issues"]),
            "dev_fit_ids": len(query_sets[project]["dev_fit_ids"]),
            "validation_queries": len(query_sets[project]["validation"]),
            "test_queries": len(query_sets[project]["test"]),
        }
        for project, data in datasets.items()
    },
}

assert run_manifest["leakage_contract"]["validation_labels_used_for_dev_training"] is False
assert run_manifest["leakage_contract"]["official_test_labels_used_for_training_or_selection"] is False
(OUTPUT_ROOT / "run_manifest.json").write_text(
    json.dumps(run_manifest, indent=2), encoding="utf-8"
)
print(json.dumps(run_manifest, indent=2))
print("PHASE 5 MANIFEST WRITTEN")


## Interpretation checklist

Phase 5 succeeds on validation only when:

```text
macro-project cross-encoder Hit@10 > macro-project dense Hit@10
```

Use MRR@10 and Recall@10 as the frozen tie-breakers. Also report:

- dense candidate Hit@50 (the reranker's coverage ceiling);
- per-project results, not only the macro average;
- paired `reranker_win` versus `dense_win` query counts;
- inference cost: every query requires up to 50 transformer forward pairs;
- hard-negative label noise as a limitation;
- that v2 directly optimizes positive-vs-hard-negative ordering per query.

If the cross-encoder loses, retain Phase 3 dense or Phase 4 RRF as the main system.
Do not force a Phase 5 win by tuning on official test. Because Phase 4 already opened
that test, obtain a new later temporal holdout before presenting Phase 5 as a fresh
confirmatory result.
